# Phase 3 — Tracking + Zone

**Goal:** Calibrate loading zone polygon. Verify BoxMOT BotSort tracks persist across 30 consecutive frames with zone overlay.

**Headless calibration:** No cv2.imshow — display frame via matplotlib, enter polygon coordinates manually in Cell 7.

Run all cells top to bottom. Do not skip Cell 1.

In [ ]:
# Cell 1 — Repo sync (ALWAYS run first)
import os, sys

repo_path = "/kaggle/working/trailer-counter"
if not os.path.exists(repo_path):
    os.system("git clone https://github.com/Rutwik1000/trailer-counter.git " + repo_path)
os.chdir(repo_path)
os.system("git fetch origin")
os.system("git reset --hard origin/master")
os.system("git log --oneline -3")

if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

In [ ]:
# Cell 2 — Install dependencies
os.system("pip install -r requirements.txt -q")
for mod in list(sys.modules.keys()):
    if mod.startswith("src"):
        del sys.modules[mod]

In [ ]:
# Cell 3 — Verify environment
import torch
print("Python:", sys.version)
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

import cv2
import numpy as np
import glob
import json
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from src.zone import LoadingZone
from src.tracker import Tracker
from src.detector import Detector
from src.preprocessor import apply_clahe
print("All src imports OK")

In [ ]:
# Cell 4 — HuggingFace auth + download OsNet weights
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, hf_hub_download

token = UserSecretsClient().get_secret("hf_token")
login(token=token, add_to_git_credential=False)
print("HF login OK")

os.makedirs("models", exist_ok=True)

osnet_dest = os.path.join("models", "osnet_x0_25_msmt17.pt")
if os.path.exists(osnet_dest):
    print("Already exists: osnet_x0_25_msmt17.pt")
else:
    hf_hub_download(
        repo_id="Zaafan/sitesense-weights",
        filename="osnet_x0_25_msmt17.pt",
        local_dir="models/",
    )
    print("Downloaded: osnet_x0_25_msmt17.pt")

assert os.path.exists(osnet_dest), "osnet_x0_25_msmt17.pt not found — check HF access"
print("OsNet weights present:", osnet_dest)

In [ ]:
# Cell 5 — Extract 30 consecutive frames from first fuxi-robot video
from datasets import load_dataset

TRACKING_FRAMES = 30
tracking_dir = os.path.join(repo_path, "data", "tracking_demo")
os.makedirs(tracking_dir, exist_ok=True)

existing = glob.glob(os.path.join(tracking_dir, "*.jpg"))
if len(existing) >= TRACKING_FRAMES:
    print(f"Already extracted {len(existing)} tracking demo frames — skipping")
else:
    ds = load_dataset("fuxi-robot/excavator-video", split="train", streaming=True)
    for sample in ds:
        video = sample["video"]
        n = min(TRACKING_FRAMES, len(video))
        for i in range(n):
            frame_np = video[i].data.permute(1, 2, 0).numpy()
            frame_bgr = cv2.cvtColor(frame_np, cv2.COLOR_RGB2BGR)
            cv2.imwrite(os.path.join(tracking_dir, f"frame_{i:03d}.jpg"), frame_bgr)
        print(f"Extracted {n} consecutive frames from first video")
        break

demo_paths = sorted(glob.glob(os.path.join(tracking_dir, "*.jpg")))
assert len(demo_paths) >= TRACKING_FRAMES, f"Expected {TRACKING_FRAMES} frames, got {len(demo_paths)}"
frame_shape = cv2.imread(demo_paths[0]).shape
print(f"Tracking demo ready: {len(demo_paths)} frames, shape={frame_shape}")

In [ ]:
# Cell 6 — Display reference frame for zone calibration
# Study the pixel coordinates of the loading area to set polygon_points in Cell 7

ref_frame = cv2.imread(demo_paths[0])
h, w = ref_frame.shape[:2]

# Draw coordinate grid every 100px
grid_frame = ref_frame.copy()
for x in range(0, w, 100):
    cv2.line(grid_frame, (x, 0), (x, h), (80, 80, 80), 1)
    cv2.putText(grid_frame, str(x), (x + 2, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)
for y in range(0, h, 100):
    cv2.line(grid_frame, (0, y), (w, y), (80, 80, 80), 1)
    cv2.putText(grid_frame, str(y), (2, y + 14), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)

plt.figure(figsize=(16, 9))
plt.imshow(cv2.cvtColor(grid_frame, cv2.COLOR_BGR2RGB))
plt.title(f"Zone Calibration Reference — frame size: {w}\u00d7{h} (W\u00d7H)")
plt.axis("off")
plt.tight_layout()
plt.show()
print(f"Frame: {w}\u00d7{h}  (width \u00d7 height)")
print("The loading zone is where the dump truck parks under the excavator arm.")
print("Enter coordinates in Cell 7 as [[x,y], ...] clockwise from top-left.")

In [ ]:
# Cell 7 — Zone calibration: enter polygon coordinates after studying Cell 6
# -------------------------------------------------------------------------
# HUMAN ACTION: Replace the example coordinates below with the actual pixel
# coordinates of the loading area corners from the frame above.
# Enter corners in order (clockwise): top-left, top-right, bottom-right, bottom-left
# -------------------------------------------------------------------------

polygon_points = [
    [320, 350],   # top-left  — ADJUST based on Cell 6 image
    [960, 350],   # top-right
    [960, 650],   # bottom-right
    [320, 650],   # bottom-left
]

# Validate and visualize
zone = LoadingZone(polygon=polygon_points)
zone_frame = zone.draw_on_frame(ref_frame)

plt.figure(figsize=(16, 9))
plt.imshow(cv2.cvtColor(zone_frame, cv2.COLOR_BGR2RGB))
plt.title(f"Zone Preview — {len(polygon_points)} point polygon (yellow outline)")
plt.axis("off")
plt.tight_layout()
plt.show()

# Save to config
zone_path = os.path.join(repo_path, "config", "loading_zone.json")
zone.save(zone_path)
print(f"Zone saved: {zone_path}")
print(f"Polygon ({len(polygon_points)} points): {polygon_points}")
print("If the yellow outline covers the loading area correctly \u2192 proceed to Cell 8.")
print("If not \u2192 re-run this cell with corrected coordinates.")

In [ ]:
# Cell 8 — Detector + Tracker + Zone integration on 30 consecutive frames
with open(os.path.join(repo_path, "config", "detector_choice.json")) as f:
    det_cfg = json.load(f)

detector = Detector(
    model_type=det_cfg["model_type"],
    weights_path=det_cfg["weights_path"],
    confidence_threshold=det_cfg["confidence_threshold"],
)
tracker = Tracker(
    reid_weights_path=os.path.join(repo_path, "models", "osnet_x0_25_msmt17.pt"),
    device="cuda" if torch.cuda.is_available() else "cpu",
)
zone = LoadingZone.load(os.path.join(repo_path, "config", "loading_zone.json"))

track_history = {}   # track_id -> list of frame indices where it was seen
annotated_frames = []

for i, path in enumerate(demo_paths[:TRACKING_FRAMES]):
    frame = cv2.imread(path)
    frame_enhanced = apply_clahe(frame)
    detections = detector.detect(frame_enhanced)
    tracks = tracker.update(detections, frame_enhanced)

    for t in tracks:
        track_history.setdefault(t["track_id"], []).append(i)

    # Annotate: draw zone + bounding boxes
    ann = zone.draw_on_frame(frame)
    for t in tracks:
        x1, y1, x2, y2 = map(int, t["bbox"])
        in_zone = zone.bbox_in_zone(t["bbox"])
        color = (0, 255, 0) if in_zone else (0, 165, 255)  # green=in, orange=out
        cv2.rectangle(ann, (x1, y1), (x2, y2), color, 2)
        label = f"ID:{t['track_id']} {'IN' if in_zone else 'OUT'}"
        cv2.putText(ann, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)
    annotated_frames.append(ann)

print(f"Processed {TRACKING_FRAMES} frames")
print(f"Unique track IDs: {sorted(track_history.keys())}")
for tid, frames in sorted(track_history.items()):
    print(f"  Track {tid}: frames {frames[0]}\u2013{frames[-1]} ({len(frames)} appearances)")

In [ ]:
# Cell 9 — Visualize annotated frames + verification assertions
n_display = min(9, len(annotated_frames))
step = max(1, len(annotated_frames) // n_display)
display = annotated_frames[::step][:n_display]

cols = 3
rows = (n_display + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(18, 6 * rows))
for ax, frame in zip(axes.flatten(), display):
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.axis("off")
for ax in axes.flatten()[n_display:]:
    ax.axis("off")
plt.suptitle("Phase 3: Tracking + Zone (green=in zone, orange=outside)", fontsize=12)
plt.tight_layout()
plt.show()

# Assertions
assert len(track_history) > 0, (
    "No tracks produced. Detector may be missing vehicles. "
    "Try lowering confidence_threshold in config/detector_choice.json to 0.1."
)
persistent = {tid: f for tid, f in track_history.items() if len(f) >= 3}
print(f"Tracks with >=3 appearances: {len(persistent)} / {len(track_history)}")
print("Phase 3 integration verified." if persistent else
      "WARNING: No persistent tracks. Zone + detector are working but tracking may need tuning.")

In [ ]:
# Cell 10 — Commit config/loading_zone.json + PROGRESS.md
import datetime

today = datetime.date.today().isoformat()
progress_path = os.path.join(repo_path, "docs", "PROGRESS.md")

with open(progress_path, "r") as f:
    content = f.read()

content = content.replace(
    "| Phase 3 — Tracking + Zone | Not started |",
    f"| Phase 3 — Tracking + Zone | Complete | {today} |",
)

n_persistent = len({tid for tid, f in track_history.items() if len(f) >= 3})
progress_entry = (
    f"\n## {today} — Phase 3 complete\n\n"
    f"**Files created:** src/tracker.py, src/zone.py, tests/test_zone.py, tests/test_tracker.py, "
    f"notebooks/03_tracking_zone.ipynb, config/loading_zone.json\n"
    f"**Tests passing:** 17 zone + 9 tracker locally (4 tracker tests ran on Kaggle)\n"
    f"**Zone:** {len(polygon_points)}-point polygon calibrated, saved to config/loading_zone.json\n"
    f"**Unique tracks over 30 frames:** {len(track_history)}\n"
    f"**Persistent tracks (>=3 frames):** {n_persistent}\n"
    f"**Blocker:** None\n"
    f"**Next:** Phase 4 — Fill Event Counter (event_counter.py state machine: "
    f"zone_entry + departure = fill event)\n"
)

insert_marker = "---\n"
insert_pos = content.find(insert_marker) + len(insert_marker)
content = content[:insert_pos] + progress_entry + content[insert_pos:]

with open(progress_path, "w") as f:
    f.write(content)

print("PROGRESS.md updated")
print(progress_entry)

from kaggle_secrets import UserSecretsClient
gh_token = UserSecretsClient().get_secret("github_token")
os.system("git config --global credential.helper store")
with open(os.path.expanduser("~/.git-credentials"), "w") as f:
    f.write("https://x-token:" + gh_token + "@github.com\n")
os.system("git config user.email 'kaggle@trailer-counter'")
os.system("git config user.name 'Kaggle Executor'")
os.system("git add config/loading_zone.json")
os.system("git add docs/PROGRESS.md")
os.system("git diff --stat HEAD")
os.system(f"git commit -m 'feat: phase 3 complete — zone calibrated ({len(polygon_points)} pts), tracking verified'")
os.system("git push origin HEAD:master")
print("Pushed to master")